# Data Input and Output (Week 2 tutorial, notebook 8 of 8)

**What this notebook covers:** reading and writing CSV and Excel files, reading a table from an HTML page, and writing to and reading from an SQL database.

**How to run it:** this notebook runs in your browser, so there is nothing to install. Click a code cell and press **Shift + Enter** to run it and move to the next one, or use **Run → Run All Cells**. Run the cells in order, because later cells use variables created earlier. The first code cell installs `lxml` (for HTML) and `openpyxl` (for Excel) into the browser, which takes a few seconds the first time.

**What to expect:** the files `example.csv`, `Excel_Sample.xlsx` and `banklist.html` are in the same folder as this notebook, so pandas finds them by name. The cells that write files change your own copy in the browser only; the files on the course website are not affected.

**Next:** this is the last notebook of the set.

## Overview

This notebook is the reference code for getting input and output. pandas can read a variety of file types using its `pd.read_` methods. Let's take a look at the most common data types:

In [1]:
%pip install lxml openpyxl
# lxml reads HTML tables and openpyxl reads and writes Excel files
import numpy as np
import pandas as pd

## CSV

### CSV Input

`example.csv` is a small comma-separated text file.

In [25]:
df = pd.read_csv('example.csv')
df

,a,b,c,d
0,0,1,2,3
1,4,5,6,7
2,8,9,10,11
3,12,13,14,15


### CSV Output

`index=False` leaves out the row labels 0 to 3, so the file keeps its four original columns.

In [24]:
df.to_csv('example.csv',index=False)

## Excel
Pandas can read and write excel files, keep in mind, this only imports data. Not formulas or images, having images or macros may cause this read_excel method to crash. 

### Excel Input

`sheet_name` picks the worksheet (older pandas versions called it `sheetname`). The first column, `Unnamed: 0`, is a row index that was saved into the file without a header; add `index_col=0` to use it as the index instead.

In [35]:
pd.read_excel('Excel_Sample.xlsx',sheet_name='Sheet1')

,Unnamed: 0,a,b,c,d
0,0,0,1,2,3
1,1,4,5,6,7
2,2,8,9,10,11
3,3,12,13,14,15


### Excel Output

`to_excel` also saves the row index as the first column, which is where the `Unnamed: 0` column above comes from.

In [33]:
df.to_excel('Excel_Sample.xlsx',sheet_name='Sheet1')

## HTML

pandas can read tables from HTML pages. It needs the `lxml` library, which the first code cell installed. On your own computer, install it once with `pip install lxml` (or `conda install lxml` if you use Anaconda).

This example originally read the FDIC Failed Bank List straight from the web (it is now at https://www.fdic.gov/bank-failures/failed-bank-list, split over several pages). Python running in the browser cannot download a page from another website this way, so this notebook reads a saved copy, `banklist.html`, with 60 of the 545 banks the list had in 2016 (the 30 most recent and the 30 oldest).

### HTML Input

Pandas read_html function will read tables off of a webpage and return a list of DataFrame objects, one for each table on the page, so `tables[0]` is the first table:

In [5]:
# on your own computer you can pass a web address instead of a file name
tables = pd.read_html('banklist.html')

In [7]:
tables[0]

,Bank Name,City,ST,CERT,Acquiring Institution,Closing Date,Updated Date,Loss Share Type,Agreement Terminated,Termination Date
0,First CornerStone Bank,King of Prussia,PA,35312,First-Citizens Bank & Trust Company,"May 6, 2016","July 12, 2016",none,NaN,NaN
1,Trust Company Bank,Memphis,TN,9956,The Bank of Fayette County,"April 29, 2016","August 4, 2016",none,NaN,NaN
2,North Milwaukee State Bank,Milwaukee,WI,20364,First-Citizens Bank & Trust Company,"March 11, 2016","June 16, 2016",none,NaN,NaN
3,Hometown National Bank,Longview,WA,35156,Twin City Bank,"October 2, 2015","April 13, 2016",none,NaN,NaN
4,The Bank of Georgia,Peachtree City,GA,35259,Fidelity Bank,"October 2, 2015","April 13, 2016",none,NaN,NaN
5,Premier Bank,Denver,CO,34112,"United Fidelity Bank, fsb","July 10, 2015","July 12, 2016",none,NaN,NaN
6,Edgebrook Bank,Chicago,IL,57772,Republic Bank of Chicago,"May 8, 2015","July 12, 2016",none,NaN,NaN
7,Doral BankEn Espanol,San Juan,PR,32102,Banco Popular de Puerto Rico,"February 27, 2015","May 13, 2015",none,NaN,NaN
8,Capitol City Bank & Trust Company,Atlanta,GA,33938,First-Citizens Bank & Trust Company,"February 13, 2015","April 21, 2015",none,NaN,NaN
9,Highland Community Bank,Chicago,IL,20290,"United Fidelity Bank, fsb","January 23, 2015","April 21, 2015",none,NaN,NaN


____

_____
_____
# SQL (Optional)

This section uses SQLite, a small database that runs inside Python, so there is no database server to set up.

The pandas.io.sql module provides a collection of query wrappers to both facilitate data retrieval and to reduce dependency on DB-specific API. Database abstraction is provided by SQLAlchemy if installed. In addition you will need a driver library for your database. Examples of such drivers are psycopg2 for PostgreSQL or pymysql for MySQL. For SQLite this is included in Python’s standard library.

If SQLAlchemy is not installed, pandas can still work with SQLite through a plain `sqlite3` connection. For more examples, see the section "SQL queries" in the pandas user guide: https://pandas.pydata.org/docs/user_guide/io.html

The key functions are:

* read_sql_table(table_name, con[, schema, ...])
    * Read SQL database table into a DataFrame.
* read_sql_query(sql, con[, index_col, ...])
    * Read SQL query into a DataFrame.
* read_sql(sql, con[, index_col, ...])
    * Read SQL query or database table into a DataFrame.
* DataFrame.to_sql(name, con[, schema, ...])
    * Write records stored in a DataFrame to a SQL database.

In [36]:
from sqlalchemy import create_engine

`create_engine('sqlite:///:memory:')` creates a temporary database in memory. `to_sql` writes `df` into a table called `data` and returns the number of rows written (4); `read_sql` reads the table back. The extra `index` column is the DataFrame's index, which `to_sql` saves by default.

In [37]:
engine = create_engine('sqlite:///:memory:')

In [40]:
# write df into a new table called 'data'
df.to_sql('data', engine)

4

In [42]:
sql_df = pd.read_sql('data',con=engine)

In [43]:
sql_df

,index,a,b,c,d
0,0,0,1,2,3
1,1,4,5,6,7
2,2,8,9,10,11
3,3,12,13,14,15


**Try it:** run `pd.read_sql('SELECT a, d FROM data WHERE a > 4', con=engine)`. What do you get?

<details><summary>Answer</summary>

Two rows: `a` = 8, `d` = 11 and `a` = 12, `d` = 15. `read_sql` also accepts an SQL query instead of a table name.

</details>

# Great Job!